Install Pyspark

In [1]:
!pip install pyspark

Dataset

In [2]:
import pandas as pd

data = {
    "transaction_id":[1,2,3,4,5,6,7,8,9,10],
    "user_id":[101,101,102,103,101,102,103,104,104,101],
    "date":[
        "2026-06-01",
        "2026-06-10",
        "2026-06-05",
        "2026-06-03",
        "2026-07-02",
        "2026-07-04",
        "2026-07-06",
        "2026-07-08",
        "2026-07-10",
        "2026-07-15"
    ],
    "category":[
        "Food",
        "Shopping",
        "Bills",
        "Food",
        "Travel",
        "Food",
        "Shopping",
        "Bills",
        "Travel",
        "Shopping"
    ],
    "amount":[
        500,
        700,
        1000,
        400,
        12000,
        900,
        500,
        1500,
        20000,
        800
    ]
}

df = pd.DataFrame(data)

df.to_csv("expenses.csv",index=False)

print(df)

   transaction_id  user_id        date  category  amount
0               1      101  2026-06-01      Food     500
1               2      101  2026-06-10  Shopping     700
2               3      102  2026-06-05     Bills    1000
3               4      103  2026-06-03      Food     400
4               5      101  2026-07-02    Travel   12000
5               6      102  2026-07-04      Food     900
6               7      103  2026-07-06  Shopping     500
7               8      104  2026-07-08     Bills    1500
8               9      104  2026-07-10    Travel   20000
9              10      101  2026-07-15  Shopping     800


Start Spark Session

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("ExpenseAnalysis") \
    .getOrCreate()

Load Data(CSV)

In [4]:
df = spark.read.csv(
    "expenses.csv",
    header=True,
    inferSchema=True
)

df.show()

+--------------+-------+----------+--------+------+
|transaction_id|user_id|      date|category|amount|
+--------------+-------+----------+--------+------+
|             1|    101|2026-06-01|    Food|   500|
|             2|    101|2026-06-10|Shopping|   700|
|             3|    102|2026-06-05|   Bills|  1000|
|             4|    103|2026-06-03|    Food|   400|
|             5|    101|2026-07-02|  Travel| 12000|
|             6|    102|2026-07-04|    Food|   900|
|             7|    103|2026-07-06|Shopping|   500|
|             8|    104|2026-07-08|   Bills|  1500|
|             9|    104|2026-07-10|  Travel| 20000|
|            10|    101|2026-07-15|Shopping|   800|
+--------------+-------+----------+--------+------+



Monthly spending per user

In [5]:
from pyspark.sql.functions import month, year, sum

monthly_spend = df.withColumn("month", month("date")) \
                  .withColumn("year", year("date")) \
                  .groupBy("user_id","year","month") \
                  .agg(sum("amount").alias("monthly_total"))

print("Monthly Spending")
monthly_spend.show()

Monthly Spending
+-------+----+-----+-------------+
|user_id|year|month|monthly_total|
+-------+----+-----+-------------+
|    102|2026|    7|          900|
|    101|2026|    7|        12800|
|    101|2026|    6|         1200|
|    102|2026|    6|         1000|
|    103|2026|    6|          400|
|    103|2026|    7|          500|
|    104|2026|    7|        21500|
+-------+----+-----+-------------+



Average spending per user

In [6]:
from pyspark.sql.functions import avg

avg_spend = df.groupBy("user_id") \
              .agg(avg("amount").alias("avg_amount"))

avg_spend.show()

+-------+----------+
|user_id|avg_amount|
+-------+----------+
|    101|    3500.0|
|    103|     450.0|
|    102|     950.0|
|    104|   10750.0|
+-------+----------+



Detect unusual spending

In [7]:
from pyspark.sql.functions import col

joined = df.join(avg_spend,"user_id")

anomalies = joined.filter(
    col("amount") > 2 * col("avg_amount")
)

print("Potential Unusual Spending")
anomalies.select(
    "transaction_id",
    "user_id",
    "date",
    "category",
    "amount",
    "avg_amount"
).show()

Potential Unusual Spending
+--------------+-------+----------+--------+------+----------+
|transaction_id|user_id|      date|category|amount|avg_amount|
+--------------+-------+----------+--------+------+----------+
|             5|    101|2026-07-02|  Travel| 12000|    3500.0|
+--------------+-------+----------+--------+------+----------+



Users with unusual spending

In [8]:
anomalies.groupBy("user_id") \
         .count() \
         .show()

+-------+-----+
|user_id|count|
+-------+-----+
|    101|    1|
+-------+-----+

